In [ ]:
import pandas as pd
import numpy as np

In [ ]:
cdr_methods =  \
['SCS',
 'A/R',
 'General forestry',
 'Agroforestry',
 'Forest management',
 'Biochar',
 'Restoration of landscapes and peats',
 'EW',
 'OAE',
 'OIF/AU',
 'DOC',
 'Algae ',
 'Blue carbon',
 'CCS',
 'BECCS',
 'DACCS',
 'CCUS',
 'General CDR',
 'Other']

mrv_topics = \
['Quantification',
 'Monitoring',
 'Quality',
 'Governance',
 'Reporting',
 'Verification',
 'External impacts',
 'General MRV']

In [ ]:
df = pd.read_excel("MRVdata_0912.xlsx", sheet_name="CodingCompleted_withoutLCA") # older version: MRVdata_0810 ?
df_filtered = df[df['Year'] != 2024]
df_included = df_filtered[df_filtered['Inclusion/exclusion'] == 'Inclusion'].copy() # new dataframe defined to exclude studies without access or excluded papers

### additonal data

def extract_CDR_method_focus(row):
    CDR_method = []
    CDR_focus = []
    
    for column in cdr_methods:  # previously defined list with columns that contain CDR methods
        if row[column] > 0:
            CDR_method.append(column)
            CDR_focus.append(row[column])
    
    return [CDR_method, CDR_focus]
# add new columns to the data frame
df_included['CDR_focus'] = df_included.apply(lambda row: extract_CDR_method_focus(row), axis=1)
df_included['CDR_method'] = df_included.CDR_focus.str[0]
df_included['CDR_focus'] = df_included.CDR_focus.str[1]


#add new column for MRV topic and study focus to the dataset
def extract_MRV_topic_focus(row):
    MRV_topic = []
    MRV_focus = []
    
    for column in mrv_topics:  # previously defined list with columns that contain MRV topics
        if row[column] > 0:
            MRV_topic.append(column)
            MRV_focus.append(row[column])
    
    return [MRV_topic, MRV_focus]
# add new columns to the data frame
df_included['MRV_focus'] = df_included.apply(lambda row: extract_MRV_topic_focus(row), axis=1)
df_included['MRV_topic'] = df_included.MRV_focus.str[0]
df_included['MRV_focus'] = df_included.MRV_focus.str[1]


### explode
# explode CDR method and focus
df_CDRexplode = df_included.explode(['CDR_method', 'CDR_focus'])

# explode MRV topic and focus
df_MRVexplode = df_included.explode(['MRV_topic', 'MRV_focus'])


### only papers on quantification
df_quant = df_MRVexplode.loc[df_MRVexplode.MRV_topic.isin(["Quantification"])]
df_quant = df_quant.explode("CDR_method")

In [ ]:
def find_modelling(string):
    if "Modelling" in string:
        try:
            return string.split(' (')[1].strip(')')
        except:
            return np.nan
df_quant["modelling type"] = df_quant["Measurement tool"].apply(find_modelling)
#df_quant["modelling type"].value_counts()

# create variable for carbon pool
df_quant["cpool_list"] = df_quant['Point of measurement/carbon pool'].str.split(', ')
df_cpool = df_quant.explode("cpool_list")
df_cpool["cpool_norm"] = df_cpool["cpool_list"].str.split(' \(').str[0]
repl = {
    "AGB": "AGB / AGB Proxy",
    "AGB Proxy": "AGB / AGB Proxy",
    "SOC": "SOC / SOC Proxy",
    "SOC Proxy": "SOC / SOC Proxy",

}
df_cpool["cpool_norm"] = df_cpool["cpool_norm"].replace(repl)


### AGB models

In [ ]:
df_cpool.loc[df_cpool["cpool_norm"]=="AGB / AGB Proxy","modelling type"].value_counts()

In [ ]:
list_agb_models = [
    "CO2FIX V.2", 
    "CBM-CFS3", 
    "FullCAM", 
    "Linear and non-linear regression models", 
    "Spatially explicit carbon bookkeeping model", 
    "CNN", 
    "Mixed-species allometric models", 
    "Non-linear tree biomass model", 
    "DayCent",
    "Forest-DNDC", 
    "Data Assimilation Linked Ecosystem Carbon model",
    "Random Forests",
    "Biome-BGC",
    "For-est"
]
agb_models = {
 'CO2FIX V.2': 'CO2FIX V.2',
 'CBM-CFS3': 'CBM-CFS3',
 'FullCAM': 'FullCAM',
 'Linear and non-linear regression models': 'own model',
 'Spatially explicit carbon bookkeeping model': 'own model',
 'CNN': 'own model',
 'Mixed-species allometric models': 'own model',
 'Non-linear tree biomass model': 'own model',
 'DayCent': 'DayCent',
 'Forest-DNDC': 'Forest-DNDC',
 'Data Assimilation Linked Ecosystem Carbon model': 'Data Assimilation Linked Ecosystem Carbon model (DALEC)',
 'Random Forests': 'own model',
 'Biome-BGC': 'Biome-BGC',
 'For-est': 'For-est'
}



In [ ]:
def get_agb_models(model_type):
    for i in list(ar_models.keys()):
        try:
            if i in model_type:
                return ar_models[i]
        except:
            return np.nan
df_cpool.loc[df_cpool["cpool_norm"]=="AGB / AGB Proxy","agb_model"] = df_cpool.loc[df_cpool["cpool_norm"]=="AGB / AGB Proxy","modelling type"].apply(get_ar_models)
df_cpool["agb_model"].value_counts()

### SOC models

In [ ]:
df_cpool.loc[df_cpool["cpool_norm"]=="SOC / SOC Proxy","modelling type"].value_counts()

In [ ]:
list_soc_models = [
    "CO2FIX V.2",
    "CBM-CFS3",
    "FullCAM",
    "Introductory Carbon Balance Model",
    "MIMICS",
    "Millenial V2",
    "Spatial Modelling",
    "CANDY, CCB, FOM-C",
    "DayCent, COMET-Farm, Cool Farm",
    "DayCent",
    "Pedotransfer function",
    "Levenberg–Marquardt error-minimisation regression model",
    "calibration model to predict SOM",
    "LUCCA",
    "non-linear regression models",
    "State-Space Model, Tarlee soil carbon model",
    "JSBACH, ISBA, LPJ-GUESS, ORCHIDEE, CLM45-GMCC, and JULES",
    "C-TOOL",
    "Data Assimilation Linked Ecosystem Carbon model"
    "CRAFT",
    "various ML models",
    "Pedotransfer function",
    "Agro-C",
    "AMG Model"   
]
'''
"Introductory Carbon Balance Model": "Introductory Carbon Balance Model (ICBM)"
"MIMICS":"MIcrobial-MIneral Carbon Stabilization (MIMICS)"
"LUCCA": "own model"
"State-Space Model, Tarlee soil carbon model":"own model"
'''
soc_models = \
{'CO2FIX V.2': 'CO2FIX V.2',
 'CBM-CFS3': 'CBM-CFS3',
 'FullCAM': 'FullCAM',
 "Introductory Carbon Balance Model": "Introductory Carbon Balance Model (ICBM)",
 "MIMICS":"MIcrobial-MIneral Carbon Stabilization (MIMICS)",
 'Millenial V2': 'Millenial V2',
 'Spatial Modelling': "own model",
 'CANDY, CCB, FOM-C': 'CANDY, CCB, FOM-C',
 'DayCent, COMET-Farm, Cool Farm': 'DayCent, COMET-Farm, Cool Farm',
 'DayCent': 'DayCent',
 'Pedotransfer function': "own model",
 'Levenberg–Marquardt error-minimisation regression model': "own model",
 'calibration model to predict SOM': "own model",
 'LUCCA': 'own model',
 'non-linear regression models': 'non-linear regression models',
 'State-Space Model, Tarlee soil carbon model': "own model",
 'JSBACH, ISBA, LPJ-GUESS, ORCHIDEE, CLM45-GMCC, and JULES': 'JSBACH, ISBA, LPJ-GUESS, ORCHIDEE, CLM45-GMCC, JULES',
 'C-TOOL': 'C-TOOL',
 'Data Assimilation Linked Ecosystem Carbon modelCRAFT': 'Data Assimilation Linked Ecosystem Carbon modelCRAFT',
 'various ML models': "own model",
 'Agro-C': 'Agro-C',
 'AMG Model': 'AMG Model'}

In [ ]:
def get_soc_models(model_type):
    for i in list(soc_models.keys()):
        try:
            if i in model_type:
                return soc_models[i]
        except:
            return np.nan
df_cpool.loc[df_cpool["cpool_norm"]=="SOC / SOC Proxy","soc_model"] = df_cpool.loc[df_cpool["cpool_norm"]=="SOC / SOC Proxy","modelling type"].apply(get_soc_models)
df_cpool["soc_model"] = df_cpool["soc_model"].str.split(', ')
df_cpool.explode("soc_model")["soc_model"].value_counts()

In [ ]:
### Remote sensing